# PLL SPICE 系列 · 整数-N CP-PLL

| 编号 | 对应课程 | 跑完看什么 |
| --- | --- | --- |
| PLL-01 | B3 · LC VCO 调谐 | 控制电压到频率的曲线、5 GHz 工作点、稳态差分幅度 |
| PLL-02 | C1 · CP 净电荷 | 40 MHz 比较周期内，边沿误差怎样变成净注入电荷 |
| PLL-03 | C2/F · 闭环对照 | 线性 VCO → SPICE 标定调谐 → 加入 CP 失配，比较捕获 |
| PLL-04 | D · 杂散及噪声 | 后续接入，同一工况分别验证 spur 和积分 jitter |

[前置实验 · VCO-01 起振与扫参](https://www.kaggle.com/code/ericzhangyin/pll-vco-01-lc-vco)

本系列使用 CPU，打开 Internet 安装锁定的 npm 依赖。Kaggle 默认从 Notebook 内附带的原始模型 ZIP 解包，无需另加数据集。本地也可以用 `PLL_LC_VCO_INPUT` 指向课程 ZIP 解包后的 `lc-vco-lab`。依次运行；原始 CSV、网表和图存在 `pll-spice-series` 输出目录。

In [ ]:
SETTINGS = {'reference_hz': 40000000, 'divide': 125, 'tune_v': [0.35, 0.5, 0.65, 0.8, 0.95, 1.1, 1.25, 1.4, 1.55], 'cap_fF': 660, 'junction_fF': 500, 'cp_uA': 100, 'loop_r_ohm': 4700, 'loop_c_pF': 100, 'initial_offset_hz': -80000000, 'run_us': 35, 'cp_mismatch': 0.05, 'reset_ns': 0.5}
LOCK_TOLERANCE_HZ = 100_000
LOCK_HOLD_US = 4

In [ ]:
# Pinned course input archive, including the SKY130 model license.
MODEL_ARCHIVE_BASE64 = (
'...'
)

In [ ]:
import base64
import hashlib
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SHA256 = {'vco.cir': '520f33cd241f2f3faf44b5d3738885cd4ec75250bf14342184b2dc25e33f3ff6', 'package-lock.json': 'e15e80c8958b1d3f780d78b5ed953c4ab167075d25184e18fabd0901cebcf404', 'models/nonfet_tt.spice': '63fdbb8e8fee2a45bf91dd5a56748a00874b741e66beec8649b024859d4dd755', 'models/sky130_fd_pr__nfet_01v8__mismatch.corner.spice': '24a3b29d7d7f26f99098811c31eb5497950a3aed520ab83768a96f35467fe818', 'models/sky130_fd_pr__nfet_01v8__tt.corner.spice': '0c03b2e5727c414942f1d2b27e19a68724fb84be90b2c73b361ad41779311130', 'models/sky130_fd_pr__nfet_01v8__tt.pm3.spice': '3d706d50cf2e57175aea78bcf27e4ee07d9f750b154a4b06e6090531364a831b'}
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
out = base / 'pll-spice-series'
out.mkdir(exist_ok=True)
tmp = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else base / '.pll-runtime'
tmp.mkdir(exist_ok=True)
input_env = os.environ.get('PLL_LC_VCO_INPUT')
candidates = [Path(input_env)] if input_env else [
    path.parent for path in Path('/kaggle/input').rglob('vco.cir')
    if (path.parent / 'package-lock.json').exists()
]
def matches(path):
    return all((path / name).is_file()
               and hashlib.sha256((path / name).read_bytes()).hexdigest() == expected
               for name, expected in SHA256.items())
source = next((path for path in candidates if matches(path)), None)
lab = tmp / 'lc-vco-lab'
if source is None and input_env:
    raise ValueError('PLL_LC_VCO_INPUT does not match the pinned course model.')
if source is None:
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(MODEL_ARCHIVE_BASE64))) as archive:
        archive.extractall(tmp)
elif source.resolve() != lab.resolve():
    shutil.copytree(source, lab, dirs_exist_ok=True, ignore=shutil.ignore_patterns('node_modules'))
assert matches(lab), 'Model integrity check failed'
env = os.environ.copy()
node = shutil.which('node')
def major(path):
    if not path:
        return 0
    return int(subprocess.check_output([path, '--version'], text=True).strip().lstrip('v').split('.')[0])
if major(node) < 22:
    node_env = tmp / 'node-22'
    node = str(node_env / 'bin' / 'node')
    if not Path(node).exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nodeenv'], check=True, timeout=180)
        subprocess.run([sys.executable, '-m', 'nodeenv', '--node=22.16.0', '--prebuilt', str(node_env)],
                       check=True, timeout=240)
    env['PATH'] = str(node_env / 'bin') + os.pathsep + env['PATH']
subprocess.run(['npm', 'ci', '--no-audit', '--no-fund'], cwd=lab, env=env, check=True, timeout=240)
print('ngspice 45.2 WASM, pinned eecircuit-engine 1.9.0')

In [ ]:
RUNNER = 'import {readFileSync, writeFileSync, mkdirSync} from \'node:fs\';\nimport {join, resolve, dirname} from \'node:path\';\n\nconst [inputArg, outArg, settingsArg] = process.argv.slice(2);\nconst input = resolve(inputArg);\nconst out = resolve(outArg);\nconst settings = JSON.parse(readFileSync(settingsArg, \'utf8\'));\nimport {Simulation} from "eecircuit-engine";\nconst sim = new Simulation();\nawait sim.start();\nmkdirSync(out, {recursive: true});\n\nfunction readDeck(path) {\n  return readFileSync(path, \'utf8\').replace(/^\\s*\\.include ["\']?([^\\s"\']+)["\']?\\s*$/gm,\n    (_, name) => readDeck(join(dirname(path), name)));\n}\n\nasync function run(id, deck, names, stop) {\n  writeFileSync(join(out, id + \'.cir\'), deck);\n  sim.setNetList(deck);\n  const result = await sim.runSim();\n  const columns = [\'time\', ...names].map(name =>\n    result?.data?.find(item => item.name.toLowerCase() === name)?.values);\n  const time = columns[0];\n  if (!time || time.at(-1) < stop * (1 - 1e-6) ||\n      columns.some(c => c?.length !== time.length || c.some(x => !Number.isFinite(x)))) {\n    throw new Error(id + \': incomplete ngspice transient\');\n  }\n  let csv = [\'time_s\', ...names].join(\',\') + \'\\n\';\n  for (let i = 0; i < time.length; i++) csv += columns.map(c => c[i]).join(\',\') + \'\\n\';\n  writeFileSync(join(out, id + \'.csv\'), csv);\n  return columns;\n}\n\nconst {reference_hz, divide, tune_v, cap_fF, junction_fF, cp_uA, loop_r_ohm,\n  loop_c_pF, initial_offset_hz, run_us, cp_mismatch, reset_ns} = settings;\nconst target = reference_hz * divide;\nif (!Number.isSafeInteger(divide) || divide < 1 || reference_hz <= 0 || cp_uA <= 0 ||\n    loop_r_ohm <= 0 || loop_c_pF <= 0 || run_us <= 0 || initial_offset_hz === 0 ||\n    cp_mismatch < 0 || cp_mismatch >= .25 || reset_ns <= .1 ||\n    tune_v.length < 5 || tune_v.some((v,i) => v < 0 || v > 1.8 || (i && v <= tune_v[i-1]))) {\n  throw new Error(\'Check PLL settings and ordered 0–1.8 V tuning grid\');\n}\nconst original = readDeck(join(input, \'vco.cir\'));\nconst points = [];\nfor (const [i, voltage] of tune_v.entries()) {\n  const label = \'pll-01-vco-\' + String(i + 1).padStart(2, \'0\');\n  const deck = original.replace(\'tankC=810f\', \'tankC=\' + cap_fF + \'f\')\n    .replace(\'tstop=120n\', \'tstop=140n\')\n    .replace(\'CN outn 0 {tankC}\', [\n      \'CN outn 0 {tankC}\',\n      \'* AC coupling gives both junctions the same independently biased DC operating point.\',\n      \'VC vctrl 0 \' + voltage,\n      \'CBP outp tune_p 5p\',\n      \'CBN outn tune_n 5p\',\n      \'RBP tune_p vctrl 100k\',\n      \'RBN tune_n vctrl 100k\',\n      \'DVP 0 tune_p DVAR\',\n      \'DVN 0 tune_n DVAR\',\n      \'.model DVAR D(Is=1e-16 Cjo=\' + junction_fF + \'f Vj=.65 M=.5)\',\n    ].join(\'\\n\'));\n  const [time, p, n] = await run(label, deck, [\'v(outp)\', \'v(outn)\', \'i(vdd)\'], 140e-9);\n  const peaks = [0, 0];\n  const edges = [];\n  for (let j = 1; j < time.length; j++) {\n    if (time[j] < 90e-9) continue;\n    const signal = p[j] - n[j];\n    const previous = p[j - 1] - n[j - 1];\n    const window = Number(time[j] >= 115e-9);\n    peaks[window] = Math.max(peaks[window], Math.abs(signal));\n    if (previous < 0 && signal >= 0)\n      edges.push(time[j - 1] - previous * (time[j] - time[j - 1]) / (signal - previous));\n  }\n  const steady = peaks[1] > .025 && Math.abs(peaks[0] - peaks[1]) / peaks[1] < .05 && edges.length > 3;\n  const hz = steady ? (edges.length - 1) / (edges.at(-1) - edges[0]) : null;\n  points.push({voltage, hz, peak_v: peaks[1], trace: label + \'.csv\'});\n  console.log(label, voltage, hz, peaks[1]);\n}\nwriteFileSync(join(out, \'pll-01-vco-metrics.json\'), JSON.stringify(points, null, 2));\n\nconst j = points.findIndex((p, i) => i > 0 && p.hz >= target && points[i - 1].hz && points[i - 1].hz < target);\nif (j < 1 || points.some((p, i) => !p.hz || (i && p.hz <= points[i - 1].hz)))\n  throw new Error(\'Provide a steady monotonic tuning grid crossing the target\');\nconst a = points[j - 1], b = points[j];\nconst gain = (b.hz - a.hz) / (b.voltage - a.voltage);\nconst lockVoltage = a.voltage + (target - a.hz) / gain;\nconst initialFrequency = target + initial_offset_hz;\nconst k = points.findIndex((p, i) => i > 0 && p.hz >= initialFrequency);\nif (k < 1 || initialFrequency <= points[0].hz) throw new Error(\'Initial frequency outside sweep\');\nconst c = points[k - 1], d = points[k];\nconst piecewiseInitial = c.voltage + (initialFrequency - c.hz) * (d.voltage - c.voltage) / (d.hz - c.hz);\nconst minVoltage = points[0].voltage, maxVoltage = points.at(-1).voltage;\nconst linearInitial = lockVoltage + initial_offset_hz / gain;\nif (linearInitial <= minVoltage || linearInitial >= maxVoltage) throw new Error(\'Linear initial voltage outside sweep\');\nconst limited = \'min(max(v(ctrl),\' + minVoltage + \'),\' + maxVoltage + \')\';\nconst interpolate = points.slice(1,-1).reduce((expr, p, i) => {\n  const slope0 = (p.hz - points[i].hz) / (p.voltage - points[i].voltage);\n  const slope1 = (points[i+2].hz - p.hz) / (points[i+2].voltage - p.voltage);\n  return expr + \'+(\' + (slope1-slope0) + \')*max(\' + limited + \'-\' + p.voltage + \',0)\';\n}, points[0].hz + \'+(\' + (points[1].hz-points[0].hz)/(points[1].voltage-points[0].voltage) + \')*(\' + limited + \'-\' + minVoltage + \')\');\n\nconst period = 1 / reference_hz;\nconst rise = 50e-12;\nconst overlap = reset_ns * 1e-9;\nconst pump = cp_uA * 1e-6;\nconst chargePoints = [];\nfor (const [i, skew] of [-1, -.5, 0, .5, 1].entries()) {\n  const dt = skew * 1e-9;\n  const start = period / 4;\n  const upStart = start + Math.max(0,-dt);\n  const dnStart = start + Math.max(0,dt);\n  const upWidth = overlap + Math.max(0,dt);\n  const dnWidth = overlap + Math.max(0,-dt);\n  const label = \'pll-02-cp-\' + String(i + 1).padStart(2,\'0\');\n  const deck = [\n    \'PLL-02 pulse-driven charge pump at the PLL reference rate\',\n    \'Iup 0 up PULSE(0 \' + pump + \' \' + upStart + \' \' + rise + \' \' + rise + \' \' + upWidth + \' \' + period + \')\',\n    \'Idn dn 0 PULSE(0 \' + pump*(1-cp_mismatch) + \' \' + dnStart + \' \' + rise + \' \' + rise + \' \' + dnWidth + \' \' + period + \')\',\n    \'Vup up ctrl 0\',\n    \'Vdn ctrl dn 0\',\n    \'Rfilter ctrl state \' + loop_r_ohm,\n    \'Cfilter state 0 \' + loop_c_pF + \'p IC=\' + lockVoltage,\n    \'.save v(ctrl) v(state) i(vup) i(vdn)\',\n    \'.tran 25p \' + period + \' 0 25p uic\',\n    \'.end\',\n  ].join(\'\\n\');\n  const [t, ctrl,, up, dn] = await run(label, deck, [\'v(ctrl)\',\'v(state)\',\'i(vup)\',\'i(vdn)\'], period);\n  let charge = 0;\n  for (let m = 1; m < t.length; m++) {\n    charge += (t[m]-t[m-1]) * (up[m]+up[m-1]-dn[m]-dn[m-1])/2;\n  }\n  chargePoints.push({skew_ns: skew, charge_fC: charge*1e15, vctrl_pp_v: Math.max(...ctrl)-Math.min(...ctrl)});\n}\nwriteFileSync(join(out, \'pll-02-cp-metrics.json\'), JSON.stringify(chargePoints,null,2));\nconst offsetA = chargePoints[2].charge_fC * 1e-15 / period;\nconst downGainA = (chargePoints[2].charge_fC-chargePoints[0].charge_fC) * 1e-6;\nconst upGainA = (chargePoints[4].charge_fC-chargePoints[2].charge_fC) * 1e-6;\n\nfor (const [mode, voltage, expression, nonideal] of [\n  [\'linear\', linearInitial, target + \'+\' + gain + \'*(\' + limited + \'-\' + lockVoltage + \')\'],\n  [\'calibrated\', piecewiseInitial, interpolate],\n  // At phase error zero the CP offset causes an instantaneous I*R step.\n  [\'calibrated-cp\', piecewiseInitial - offsetA*loop_r_ohm, interpolate, true],\n]) {\n  const stop = run_us * 1e-6;\n  const deck = [\n    \'PLL-03 SKY130-calibrated averaged integer-N CP-PLL: \' + mode,\n    \'* Frequency output node is scaled to GHz. Phase-error node is in reference cycles.\',\n    \'Bfreq freq 0 V={(\' + expression + \')/1e9}\',\n    \'Bphase 0 phase I={\' + reference_hz + \'-v(freq)*1e9/\' + divide + \'}\',\n    \'Cphase phase 0 1 IC=0\',\n    \'Bpump 0 ctrl I={\' + (nonideal\n      ? offsetA + \'+(\' + upGainA + \')*min(max(v(phase),0),1)+(\' + downGainA + \')*max(min(v(phase),0),-1)\'\n      : cp_uA + \'u*min(max(v(phase),-1),1)\') + \'}\',\n    \'Rfilter ctrl state \' + loop_r_ohm,\n    \'Cfilter state 0 \' + loop_c_pF + \'p IC=\' + voltage,\n    \'.save v(freq) v(phase) v(ctrl) v(state)\',\n    \'.tran 1n \' + stop + \' 0 1n uic\',\n    \'.end\',\n  ].join(\'\\n\');\n  await run(\'pll-03-loop-\' + mode, deck, [\'v(freq)\', \'v(phase)\', \'v(ctrl)\', \'v(state)\'], stop);\n}\nwriteFileSync(join(out, \'pll-03-loop-config.json\'), JSON.stringify({\n  target_hz: target, lock_voltage_v: lockVoltage, gain_hz_v: gain,\n  cp_offset_a: offsetA, cp_up_gain_a: upGainA, cp_down_gain_a: downGainA,\n  linear_initial_v: linearInitial, calibrated_initial_v: piecewiseInitial, settings,\n}, null, 2));\n'
runner = lab / 'pll_spice_series.mjs'
runner.write_text(RUNNER, encoding='utf-8')
(out / 'settings.json').write_text(json.dumps(SETTINGS, indent=2), encoding='utf-8')
with (out / 'ngspice-run.log').open('w') as log:
    result = subprocess.run([node, str(runner), str(lab), str(out), str(out / 'settings.json')],
                            cwd=lab, env=env, stdout=log, stderr=subprocess.STDOUT, timeout=600)
print((out / 'ngspice-run.log').read_text())
result.check_returncode()

## PLL-01 · VCO 调谐
SKY130 TT、27 °C、1.8 V 振荡核心沿用起振实验网表。两只交流耦合的结电容使用教学二极管模型，偏置电压直接进入 SPICE 网表。稳态频率由差分波形同向过零点测得。5 GHz 交点使用相邻两点插值，图线是点间连接。

In [ ]:
vco = pd.read_json(out / 'pll-01-vco-metrics.json')
cfg = json.loads((out / 'pll-03-loop-config.json').read_text())
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(vco.voltage, vco.hz / 1e9, 'o-', label='SPICE sweep')
ax.axhline(cfg['target_hz']/1e9, color='tab:orange', linestyle='--', label='Target')
ax.axvline(cfg['lock_voltage_v'], linestyle=':', color='tab:green', label='Lock bias')
ax.set(xlabel='VCTRL (V)', ylabel='Frequency (GHz)', title='PLL-01 | LC VCO tuning')
ax.legend()
ax.grid(alpha=.25)
fig.savefig(out / 'pll-01-vco.png', dpi=150, bbox_inches='tight')
plt.show()
display(vco.assign(f_GHz=vco.hz / 1e9)[['voltage','f_GHz','peak_v']].round(5))
print(f"Lock bias: {cfg['lock_voltage_v']:.5f} V, local gain: {cfg['gain_hz_v']/1e6:.2f} MHz/V")

## PLL-02 · PFD 边沿误差转成 CP 净电荷
用脉冲源施加 UP/DN，以电流源近似 CP。正时间差延长 UP，负时间差延长 DN。带失配时，即使边沿同时到达，重叠脉冲也可能留下净电荷。净电荷和两侧斜率进入下一节的平均电荷泵模型。图中的理想电流源没有随输出电压变化的限流和合规范围。

In [ ]:
cp = pd.read_json(out / 'pll-02-cp-metrics.json')
cp_wave = pd.read_csv(out / 'pll-02-cp-03.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout='constrained')
axes[0].plot(cp.skew_ns, cp.charge_fC, 'o-')
axes[0].axhline(0, color='gray', linestyle=':')
axes[0].set(xlabel='Edge skew (ns)', ylabel='Net charge (fC)')
for name, label in [('i(vup)', 'UP'), ('i(vdn)', 'DN')]:
    axes[1].plot(cp_wave.time_s*1e9, cp_wave[name]*1e6, label=label)
axes[1].set(xlabel='Time (ns)', ylabel='Current (uA)', xlim=(5.5,8))
axes[1].legend()
fig.suptitle('PLL-02 | Charge pump pulse response')
fig.savefig(out / 'pll-02-cp.png', dpi=150)
plt.show()
display(cp.round(5))

## PLL-03 · 闭环接线和参数替换
同一 40 MHz 参考和 N=125，目标 5 GHz，三组从相同输出频差开始。先看频差，再对照控制电压，最后看 CP 失配让锁定相位偏到哪里。闭环网表用平均 PFD/CP 和相位状态完成反馈，时域由 ngspice 求解。它适合做这版捕获对照。逐边沿 PFD/分频、cycle slip、参考杂散和相噪属于后续工作。

In [ ]:
rows = []
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, layout='constrained')
for mode in ['linear','calibrated','calibrated-cp']:
    trace = pd.read_csv(out / f'pll-03-loop-{mode}.csv')
    time = trace.time_s.to_numpy()
    error = trace['v(freq)'].to_numpy() * 1e9 - cfg['target_hz']
    assert abs(error[0] - SETTINGS['initial_offset_hz']) < 5e4, mode
    assert time[-1] >= SETTINGS['run_us'] * 1e-6 * (1-1e-6), mode
    bad = np.flatnonzero(np.abs(error) > LOCK_TOLERANCE_HZ)
    entries = np.r_[0, bad + 1]
    ends = np.r_[bad-1, len(time)-1]
    eligible = [time[a] for a,b in zip(entries, ends) if a < b
                and time[b] - time[a] >= LOCK_HOLD_US*1e-6]
    lock = None if not eligible else eligible[0] * 1e6
    axes[0].plot(time*1e6, error/1e6, label=mode)
    axes[1].plot(time*1e6, trace['v(ctrl)'], label=mode)
    rows.append(dict(case=mode, first_held_in_band_us=lock,
                     end_error_hz=error[-1], end_vctrl_v=trace['v(ctrl)'].iloc[-1],
                     end_phase_deg=360*trace['v(phase)'].iloc[-1]))
axes[0].legend()
axes[0].set(ylabel='Output error (MHz)', title='PLL-03 | Averaged closed-loop capture')
axes[1].set(xlabel='Time (us)', ylabel='VCTRL (V)')
for ax in axes:
    ax.grid(alpha=.25)
fig.savefig(out / 'pll-03-loop.png', dpi=150)
plt.show()
summary = pd.DataFrame(rows)
summary.to_csv(out / 'pll-03-loop-metrics.csv', index=False)
print(f"Lock criterion: ±{LOCK_TOLERANCE_HZ:,} Hz for {LOCK_HOLD_US} us")
display(summary.round(5))